<a href="https://colab.research.google.com/github/Panfordd/lab-4-llm-decision-support/blob/main/Lab%204_llm_support_decision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


# Section 1 — Talking to an LLM Programmatically

**Part 1.1 — Your first API call**

In [3]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
          temperature=0.7, max_tokens=500):
  response = client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user",   "content": user_prompt},
      ],
      temperature=temperature,
      max_tokens=max_tokens,
  )
  return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
response=ask_llm("What is a balance?")
print(response)

# TODO: Print response.usage as well — how many tokens did your call consume?
response = client.chat.completions.create(
        model=MODEL,
       messages=[
           {"role": "system", "content": "You are a helpful assistant"},
           {"role": "user",   "content": "What is a balance?"},
       ],
       temperature=0.7,
       max_tokens=500,
   )
print(response.usage)

A balance, in general, refers to a state of equilibrium or stability between different elements, forces, or aspects. It can be applied to various contexts, including:

1. **Physics**: A balance is a device used to measure the weight or mass of an object by comparing it to a standard unit of weight. It works by balancing the weight of the object against a known weight, using a fulcrum or pivot point.
2. **Life and well-being**: Balance refers to a state of harmony and equilibrium between different aspects of life, such as work, leisure, relationships, and personal growth. Achieving a balance between these areas can lead to overall well-being and happiness.
3. **Finance**: A balance can refer to the amount of money in a bank account or the state of one's financial situation, where income and expenses are balanced to maintain financial stability.
4. **Nature**: Balance can refer to the equilibrium of ecosystems, where the relationships between living organisms and their environment are in

**Student Reasoning — Anatomy of a call** 1. What is the difference between the system and user roles? Give an example of something that belongs in each. 2. What is a token, roughly? Why do API providers bill per token rather than per request?

**Part 1.2 — Temperature: the randomness dial**

In [4]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
for temperature in [0.0, 1.2]:
  print("\nTemperature: " + str(temperature))
  for i in range(5):
    response = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=temperature)
    print("Response " + str(i + 1) + ": " + response)
    print()

# TODO: Print all 10 answers, grouped by temperature.


Temperature: 0.0
Response 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to gather and save their earnings.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could be attractive to market traders who need to access their savings easily.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, which could convey a sense of community and mutual support among market traders.
6. **Kae Dwa**: "Kae Dwa" means "good savings" or "profitable savings" in the Akan language, which could appeal to market traders who want to save and grow th

**Student Reasoning — Temperature **What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

# Section 2 — The Dataset: Loan Application Letters

In [5]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


# Section 3 — Prompt Engineering for the Decision Support System

# Part 3.1 — Component 1: Summarization

In [8]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1= "Summarize this:"

summaries_v1 = {}
for letter_id in ["L002", "L006"]:
  letter_text = LETTERS[letter_id]

  user_prompt = SUMMARY_PROMPT_V1 + "\n\n" + letter_text

  answer = ask_llm(user_prompt)
  summaries_v1[letter_id] = answer


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_PROMPT_V2 = """You are an assistant to a microfinance loan officer.
                       Summarise the applicant's loan request in a factual and neutral way

                       Requirements:
                       1. Use only information stated in the loan application
                       2. Do not invent or assume any details
                       3. Keep the summary between 3-4 sentences
                       Include important facts about the applicant, loan amount, purpose,repayment information, and financial situation where available"""

summaries_v2 = {}
for letter_id in["L002", "L006"]:
  letter_text = LETTERS[letter_id]

  user_prompt = SUMMARY_PROMPT_V2 + "\n\n"+ letter_text

  answer = ask_llm(user_prompt,
                 system_prompt = SUMMARY_PROMPT_V2,
                 temperature =0)
  summaries_v2[letter_id] = answer


# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("\n--- Comparing V1 and V2 Summaries ---")
for letter_id in ["L002", "L006"]:
    print(f"\n{letter_id}:")
    print(f"V1 Summary:\n{summaries_v1[letter_id]}\n")
    print(f"V2 Summary:\n{summaries_v2[letter_id]}\n")


L002 V1 SUMMARY
Kwame Boateng, a commercial driver in Kumasi, is requesting GHS 25,000 to repair his vehicle's engine and pay off debts. He's experiencing a slow business period but expects it to improve after the festive season, and promises to repay the loan as soon as possible, despite not having collateral.

L006 V1 SUMMARY
Kofi, a 22-year-old, is requesting a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his businesses are successful, relying on his trustworthiness as assurance.

L002 V2 SUMMARY
Kwame Boateng, a commercial driver from Kumasi, has applied for a loan of GHS 25,000. The purpose of the loan is to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season, and although he does not have a specific repayment plan, he in